In [2]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models.vision_transformer import vit_b_16
from PIL import Image
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from tqdm import tqdm

# ------------------------
# Config
# ------------------------
sample_name = "XETG00248__0010314__NL518C__20240411__220137"
image_dir = f"data/demo/images/output-{sample_name}"
model_path = "output/best_model.pt"
h5ad_path = f"data/gene_expression/output-{sample_name}.h5ad"
predict_out_dir = "predict"
real_out_dir = "real"
os.makedirs(predict_out_dir, exist_ok=True)
os.makedirs(real_out_dir, exist_ok=True)

output_embedding_csv = f"{predict_out_dir}/{sample_name}.csv"
output_clustered_csv = f"{predict_out_dir}/{sample_name}_clustered.csv"
output_pred_plot = f"{predict_out_dir}/umap_leiden_pred.pdf"
output_real_plot = f"{real_out_dir}/umap_leiden_real.pdf"

num_genes = 372  # 👈 match your shared genes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------
# Model Definition
# ------------------------
class Image2Transcripts(nn.Module):
    def __init__(self, num_genes, embed_dim=768):
        super().__init__()
        self.image_encoder = vit_b_16(pretrained=True)
        self.image_encoder.heads = nn.Identity()
        self.gene_encoder = nn.Sequential(
            nn.Linear(num_genes, 512),
            nn.ReLU(),
            nn.Linear(512, embed_dim),
        )

    def forward(self, image, gene=None):
        return self.image_encoder(image)

# ------------------------
# Extract Image Embeddings
# ------------------------
model = Image2Transcripts(num_genes=num_genes).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

results = []
with torch.no_grad():
    for fname in tqdm(os.listdir(image_dir), desc="🔍 Extracting Embeddings"):
        if not fname.endswith(".png"):
            continue
        cell_id = os.path.splitext(fname)[0]
        img_path = os.path.join(image_dir, fname)
        try:
            image = Image.open(img_path).convert("RGB")
            image_tensor = transform(image).unsqueeze(0).to(device)
            embedding = model(image_tensor).squeeze(0).cpu().numpy()
            results.append((cell_id, *embedding))
        except Exception as e:
            print(f"⚠️ Error processing {fname}: {e}")

# Save to CSV
embed_dim = len(results[0]) - 1
columns = ["cell_id"] + [f"dim_{i}" for i in range(embed_dim)]
df_pred = pd.DataFrame(results, columns=columns)
df_pred.to_csv(output_embedding_csv, index=False)
print(f"✅ Saved predicted embeddings: {output_embedding_csv}")

# ------------------------
# Leiden on Predicted Embeddings
# ------------------------
adata_pred = sc.AnnData(df_pred.drop(columns=["cell_id"]).values)
adata_pred.obs["cell_id"] = df_pred["cell_id"].values
sc.pp.scale(adata_pred)
sc.pp.neighbors(adata_pred, n_neighbors=15, metric='cosine')
sc.tl.umap(adata_pred)
sc.tl.leiden(adata_pred, resolution=0.5)

# Save clustering CSV
df_clustered_pred = pd.DataFrame({
    "cell_id": adata_pred.obs["cell_id"],
    "leiden": adata_pred.obs["leiden"],
    "UMAP_1": adata_pred.obsm["X_umap"][:, 0],
    "UMAP_2": adata_pred.obsm["X_umap"][:, 1]
})
df_clustered_pred.to_csv(output_clustered_csv, index=False)
sc.pl.umap(adata_pred, color="leiden", save="_pred.pdf", show=False)
os.rename("figures/umap_pred.pdf", output_pred_plot)

print(f"✅ Predicted UMAP & Leiden saved to: {output_pred_plot}")

# ------------------------
# Real Data: UMAP + Leiden
# ------------------------
adata_real = sc.read_h5ad(h5ad_path)
sc.pp.pca(adata_real, n_comps=50)
sc.pp.neighbors(adata_real, n_neighbors=15, use_rep='X_pca')
sc.tl.umap(adata_real, min_dist=0.1, random_state=42)
sc.tl.leiden(adata_real, resolution=0.5, random_state=42)
sc.pl.umap(adata_real, color="leiden", save="_real.pdf", show=False)
os.rename("figures/umap_real.pdf", output_real_plot)

print(f"✅ Real UMAP & Leiden saved to: {output_real_plot}")

/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
🔍 Extracting Embeddings: 100%|██████████| 15771/15771 [09:22<00:00, 28.02it/s]


✅ Saved predicted embeddings: predict/XETG00248__0010314__NL518C__20240411__220137.csv


/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/scanpy/tools/_utils.py:41: UserWarning: You’re trying to run this on 768 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  warnings.warn(
/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_30530/2715913619.py:92: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_pred, resolution=0.5)


✅ Predicted UMAP & Leiden saved to: predict/umap_leiden_pred.pdf
✅ Real UMAP & Leiden saved to: real/umap_leiden_real.pdf
